# 12. External model training: CropHarvest + Sen4AgriNet/FranceCrops Prototype

| Table | Source datasets |
|---|---|
| `cropharvest/cropharvest_features` | CropHarvest (France, Madagascar, Martinique, Réunion, Senegal; 2019/2021) |
| `francecrops_s4a/partitioned_features` | Sen4AgriNet 2020 (carrot, potato) and FranceCrops 2022 (cabbage) |

**Known confound:** in the France table every cabbage row is FranceCrops 2022 and every carrot/potato row is Sen4AgriNet 2020. A France-only model can partly learn source/year instead of crop. The cross-source experiments in Step 4 check this.

| Step | What happens |
|---|---|
| 1 | Download (or reuse) both tables, harmonize, recompute SAVI and shape features, link parcels sharing a pixel, lock a 20% parcel test set |
| 2 | Grouped 5-fold cross-validation on development parcels: 3 feature designs × 3 models; then spatial-block CV and a no-weighting ablation |
| 3 | Fit the CV-selected configuration on all development parcels, score the locked test set once |
| 4 | Cross-source transfer: CropHarvest → France and France → CropHarvest |

In [ ]:
from pathlib import Path
import json
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from crop_pipeline.features import experiments as feature_experiments
from crop_pipeline.training import external_combined as ext

# The transfer tests score sources that lack some crops, which sklearn reports on every call
warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
settings_path = Path("settings.local.json")
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
earth_engine_project = local_settings.get("earth_engine_project", "thesis-18")
report_folder = Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).expanduser().resolve()

run_name = "external_ch_s4a_v1"  # Use a new name if the input tables change; the test split is locked.
refresh_inputs = False            # True re-downloads both tables from Earth Engine.
output_folder = report_folder.parent / "analysis" / run_name
figure_folder = output_folder / "figures"
figure_folder.mkdir(parents=True, exist_ok=True)
pd.set_option("display.precision", 3)
print("Output folder:", output_folder)

## 1. Inputs and locked test split

Earth Engine is contacted only when a cached table is missing or `refresh_inputs = True`. Parcel identities are prefixed by source. Parcels from different sources that share a pixel location (within 10 m) are joined into one split unit so they never fall on both sides of a split. 20% of split units in every crop × source stratum are reserved for testing; the split is saved and must match on reruns.

SAVI and the temporal-shape features are recomputed the same way for every source. CropHarvest saved no window dates, so all shapes use nominal half-month spacing.

In [ ]:
input_folder = output_folder / "inputs"
needs_download = refresh_inputs or not all((input_folder / f"{table}.csv").is_file() for table in ext.ASSETS)
earth_engine = None
if needs_download:
    import ee
    ee.Initialize(project=earth_engine_project)
    earth_engine = ee
tables, input_manifest = ext.cached_tables(earth_engine, earth_engine_project, input_folder, refresh_inputs)
for table, record in input_manifest["tables"].items():
    print(f"{table}: {record['rows']:,} rows from {record['asset_id']}")

data = ext.add_derived_features(ext.harmonize(tables))
data, overlap_audit = ext.link_split_units(data)
data = ext.add_spatial_blocks(data)
data = ext.lock_test_split(data, output_folder / "test_split.csv")
data.to_csv(output_folder / "combined_features.csv", index=False)

print(f"Parcel groups sharing a pixel location: {len(overlap_audit)} pairs")
if len(overlap_audit):
    display(overlap_audit.head(20))
print(f"Split units: {data['split_unit'].nunique()}, spatial blocks: {data['spatial_block'].nunique()}")
counts = ext.split_counts(data)
display(counts.pivot_table(index=["crop", "domain"], columns="partition", values=["parcels", "pixels"], fill_value=0))

## 2. Cross-validation on development parcels

Folds are grouped by split unit and stratified by crop, so a parcel's pixels never appear in both the fitted and scored folds. Each crop gets equal total training weight, shared equally among its parcels.

| Design | Predictors |
|---|---|
| `interval_only` | 8 half-months × (B3, B4, B5, B8, B11, NDVI, NDRE, NDMI, SAVI) = 72 |
| `interval_and_shape` | the 72 above + 48 temporal-shape features |
| `shape_only` | 48 temporal-shape features |

Models: Random Forest (1,000 trees, seed 42), histogram gradient boosting, and standardized logistic regression. The configuration with the highest mean macro-F1 is selected **before** the test set is touched.

Two checks on the selected configuration follow: **spatial-block CV** (10 km cells, so neighbouring parcels cannot be in both folds) and **no weighting**.

In [ ]:
development = data[data["partition"] == "development"].reset_index(drop=True)
designs = list(feature_experiments.FEATURE_DESIGNS)
cv_results = ext.cross_validate(development, designs, ext.MODEL_NAMES)
cv_summary = ext.summarize_cv(cv_results)
selection = ext.select_configuration(cv_summary)
print("Selected (CV only):", selection)

checks = pd.concat([
    ext.cross_validate(development, [selection["design"]], [selection["model"]], group_column="spatial_block"),
    ext.cross_validate(development, [selection["design"]], [selection["model"]], weighted=False),
])
cv_results = pd.concat([cv_results, checks], ignore_index=True)
cv_summary = ext.summarize_cv(cv_results)
ext.save_results(output_folder, cv_results=cv_results, cv_summary=cv_summary, selection=selection)
display(cv_summary[["grouping", "weighted", "design", "model", "macro_f1_mean", "macro_f1_std",
                    "parcel_macro_f1_mean", "recall_carrot_mean", "recall_cabbage_mean", "recall_potato_mean"]])

In [ ]:
main = cv_summary[(cv_summary["grouping"] == "split_unit") & cv_summary["weighted"]]
figure, axis = plt.subplots(figsize=(8, 4))
width = 0.25
for offset, model in enumerate(ext.MODEL_NAMES):
    rows = main[main["model"] == model].set_index("design").reindex(designs)
    axis.bar(np.arange(len(designs)) + (offset - 1) * width, rows["macro_f1_mean"], width,
             yerr=rows["macro_f1_std"], capsize=3, label=model.replace("_", " "))
axis.set_xticks(np.arange(len(designs)), [design.replace("_", " ") for design in designs])
axis.set_ylabel("Macro-F1 (5-fold grouped CV)")
axis.set_ylim(0, 1)
axis.legend(frameon=False, loc="lower right")
axis.set_title("Development cross-validation (mean ± sd across folds)")
figure.tight_layout()
figure.savefig(figure_folder / "cv_comparison.png", dpi=150)
plt.show()

## 3. Locked test set

The selected configuration is refitted on all development parcels and scored once on the held-out parcels. The 95% interval resamples whole test parcels within each crop. Permutation importance shows how much test macro-F1 drops when one predictor is shuffled.

In [ ]:
test_metrics, importances = ext.final_test(data, selection, output_folder)
ext.save_results(output_folder, test_metrics=test_metrics)
bootstrap = test_metrics["bootstrap"]
print(f"Selected: {selection['design']} / {selection['model']}")
print(f"Test pixels={test_metrics['pixels']:,}, parcels={test_metrics['parcels']}")
print(f"Accuracy={test_metrics['accuracy']:.3f}  macro-F1={test_metrics['macro_f1']:.3f} "
      f"(95% parcel bootstrap {bootstrap['macro_f1_low']:.3f}-{bootstrap['macro_f1_high']:.3f})  "
      f"parcel macro-F1={test_metrics['parcel_macro_f1']:.3f}")
display(pd.DataFrame(test_metrics["per_crop"]).T)
display(pd.DataFrame({domain: {key: metrics[key] for key in ("pixels", "parcels", "accuracy", "parcel_accuracy")}
                      for domain, metrics in test_metrics["by_domain"].items()}).T)

figure, (left, right) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1, 1.4]})
matrix = np.asarray(test_metrics["confusion_matrix"])
shares = matrix / matrix.sum(axis=1, keepdims=True)
left.imshow(shares, cmap="Blues", vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        left.text(j, i, f"{matrix[i, j]}\n{shares[i, j]:.0%}", ha="center", va="center",
                  color="white" if shares[i, j] > 0.5 else "black")
left.set_xticks(range(3), ext.CLASSES)
left.set_yticks(range(3), ext.CLASSES)
left.set_xlabel("Predicted")
left.set_ylabel("Source label")
left.set_title("Test confusion (pixels)")
top = importances.head(15).iloc[::-1]
right.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"], color="#4C78A8")
right.set_xlabel("Drop in test macro-F1 when shuffled")
right.set_title("Top 15 permutation importances")
figure.tight_layout()
figure.savefig(figure_folder / "test_confusion_importance.png", dpi=150)
plt.show()

## 4. Cross-source transfer

The selected configuration is trained on one source family and tested on the other, with no shared rows. A large drop from the Step 3 score means the pooled model leans on source-specific patterns. This is the closest available proxy for moving to a new region such as Paoay, but it is not evidence of Philippine accuracy.

In [ ]:
transfer = ext.transfer_experiments(data, selection)
ext.save_results(output_folder, transfer_metrics=transfer)
summary = pd.DataFrame({
    direction: {"pixels": metrics["pixels"], "parcels": metrics["parcels"], "accuracy": metrics["accuracy"],
                "macro_f1": metrics["macro_f1"], "parcel_macro_f1": metrics["parcel_macro_f1"],
                **{f"recall_{crop}": metrics["per_crop"][crop]["recall"] for crop in ext.CLASSES}}
    for direction, metrics in transfer.items()
}).T
summary.loc["pooled_test (Step 3)"] = {"pixels": test_metrics["pixels"], "parcels": test_metrics["parcels"],
    "accuracy": test_metrics["accuracy"], "macro_f1": test_metrics["macro_f1"],
    "parcel_macro_f1": test_metrics["parcel_macro_f1"],
    **{f"recall_{crop}": test_metrics["per_crop"][crop]["recall"] for crop in ext.CLASSES}}
display(summary)
for direction, metrics in transfer.items():
    print(direction)
    display(pd.DataFrame({region: {"pixels": m["pixels"], "accuracy": m["accuracy"],
                                   **{f"recall_{crop}": m["per_crop"][crop]["recall"] for crop in ext.CLASSES}}
                          for region, m in metrics["by_region"].items()}).T)

## 5. Conclusions

- **Pooled test score:** the CV-selected model (temporal-shape features, logistic regression; CV macro-F1 0.558 ± 0.092) scores **macro-F1 0.546** on the locked test set (95% parcel bootstrap 0.417-0.670; 634 pixels from 58 parcels). Potato is the weakest crop (F1 0.47, against 0.60 for carrot and 0.58 for cabbage).
- **Transfer drop:** a model trained on one source family and tested on the other does much worse. CropHarvest → Sen4AgriNet/FranceCrops falls to 0.439 (-0.11) and Sen4AgriNet/FranceCrops → CropHarvest to 0.299 (-0.25). In the second direction carrot recall falls to 0.15, and in Senegal and Martinique no carrot or potato pixel is recognised. The pooled 0.546 therefore depends on having the test region's own source in training.
- **Source confound:** in the Sen4AgriNet/FranceCrops table every cabbage row is FranceCrops 2022 and every carrot/potato row is Sen4AgriNet 2020, so part of what that table teaches is source and year, not crop. The test results show this: the 4 FranceCrops test parcels score 0.79 accuracy, against 0.45 for CropHarvest. The confound cannot be removed with these tables; the cross-source tests are the only check.
- **For Atok:** the transfer drop is the expected direction when moving to a new region. Notebook 18 tests these tables on unseen Atok fields.